<div style="background-color:#f0f2f6; padding:15px; border-radius:10px; border-left: 5px solid #4CAF50;">
    <h3 style="color:#2c3e50; margin:0;">1. Install Libraries</h3>
    <p style="color:#555; margin-top:5px;">
        Installing the necessary Python libraries. <b>NeuralForecast</b> is the main library for time-series forecasting, and <b>Torch</b> (PyTorch) is the deep learning backend required to run the models.
    </p>
</div>

In [6]:
!pip install neuralforecast -q
!pip install torch -q

<div style="background-color:#f0f2f6; padding:15px; border-radius:10px; border-left: 5px solid #4CAF50;">
    <h3 style="color:#2c3e50; margin:0;">2. Import Libraries & Setup</h3>
    <p style="color:#555; margin-top:5px;">
        Importing the necessary libraries for data manipulation (Pandas, Numpy), visualization (Matplotlib, Seaborn), and machine learning.
        We also configure the plotting style and suppress warnings for cleaner output.
    </p>
</div>

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import json
from datetime import datetime
import time

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from neuralforecast import NeuralForecast
from neuralforecast.models import PatchTST
from neuralforecast.losses.pytorch import MAE

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


<div style="background-color:#f0f2f6; padding:15px; border-radius:10px; border-left: 5px solid #4CAF50;">
    <h3 style="color:#2c3e50; margin:0;">3. Check GPU Availability</h3>
    <p style="color:#555; margin-top:5px;">
        Checking if a GPU is available for faster training. If not, it suggests switching the runtime type.
    </p>
</div>

In [8]:
import torch

if torch.cuda.is_available():
    print("✅ GPU is available!")
    print(f"📊 Device Name: {torch.cuda.get_device_name(0)}")
    print(f"💾 Available Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("❌ GPU is NOT available!")
    print("⚠️ Training will be very slow.")
    print("💡 Go to Runtime → Change runtime type → GPU → Save")

✅ GPU is available!
📊 Device Name: Tesla T4
💾 Available Memory: 14.6 GB


<div style="background-color:#f0f2f6; padding:15px; border-radius:10px; border-left: 5px solid #4CAF50;">
    <h3 style="color:#2c3e50; margin:0;">4. Mount Google Drive</h3>
    <p style="color:#555; margin-top:5px;">
        Connecting to Google Drive to access the dataset and save the trained model artifacts directly to the cloud storage.
    </p>
</div>

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


<div style="background-color:#f0f2f6; padding:15px; border-radius:10px; border-left: 5px solid #4CAF50;">
    <h3 style="color:#2c3e50; margin:0;">5. Load Dataset</h3>
    <p style="color:#555; margin-top:5px;">
        Reading the processed dataset from the specified path in Google Drive.
        We parse the 'Date' column as datetime objects and set it as the index for time-series handling.
        Basic statistics about the dataset (rows, columns, time range) are printed for verification.
    </p>
</div>

In [10]:
data_path = '/content/drive/MyDrive/Processed_Data/modelled_dataset.csv'

df = pd.read_csv(data_path, parse_dates=['Date'], index_col='Date')

print(f"✅ Data loaded successfully!")
print(f"📊 Number of Rows: {len(df):,}")
print(f"📋 Number of Columns: {len(df.columns)}")
print(f"📅 Time Period: From {df.index.min().date()} to {df.index.max().date()}")

df.head()

✅ Data loaded successfully!
📊 Number of Rows: 734
📋 Number of Columns: 81
📅 Time Period: From 2018-09-20 to 2020-09-22


,Bag,Ballerinas,Belt,Bikini top,Blazer,Blouse,Bodysuit,Boots,Bra,Bracelet,...,Underwear bottom,Underwear set,Vest top,Wallet,Watch,Wedge,day_of_week,month,day_of_year,is_weekend
Date,,,,,,,,,,,,,,,,,,,,,
2018-09-20,377.0,82.0,244.0,535.0,684.0,2571.0,207.0,448.0,1727.0,23.0,...,1335.0,8.0,1240.0,0.0,17.0,25.0,3,9,263,0
2018-09-21,346.0,87.0,229.0,475.0,678.0,2511.0,173.0,424.0,1614.0,13.0,...,1210.0,1.0,1153.0,2.0,7.0,21.0,4,9,264,0
2018-09-22,105.0,18.0,87.0,125.0,290.0,921.0,49.0,145.0,599.0,8.0,...,564.0,0.0,428.0,1.0,3.0,9.0,5,9,265,1
2018-09-23,297.0,74.0,217.0,474.0,618.0,2440.0,189.0,478.0,1538.0,10.0,...,1267.0,2.0,1418.0,2.0,17.0,35.0,6,9,266,1
2018-09-24,292.0,68.0,197.0,492.0,637.0,2244.0,175.0,408.0,1404.0,14.0,...,1209.0,3.0,1196.0,2.0,15.0,29.0,0,9,267,0


<div style="background-color:#f0f2f6; padding:15px; border-radius:10px; border-left: 5px solid #4CAF50;">
    <h3 style="color:#2c3e50; margin:0;">6. Data Inspection & Statistics</h3>
    <p style="color:#555; margin-top:5px;">
        Checking the data structure using <code>df.info()</code> to see data types and non-null counts.
        We also explicitly check for missing values and display descriptive statistics (mean, min, max, std) for the numerical columns.
    </p>
</div>

In [11]:
print("🔎 Data Information:")
print("="*60)
df.info()

print("\n" + "="*60)
print("📊 Missing Values:")
missing = df.isnull().sum()
if missing.sum() == 0:
    print("✅ No missing values found!")
else:
    print(missing[missing > 0])

print("\n" + "="*60)
print("📈 Descriptive Statistics:")
df.describe()

🔎 Data Information:
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 734 entries, 2018-09-20 to 2020-09-22
Data columns (total 81 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Bag                       734 non-null    float64
 1   Ballerinas                734 non-null    float64
 2   Belt                      734 non-null    float64
 3   Bikini top                734 non-null    float64
 4   Blazer                    734 non-null    float64
 5   Blouse                    734 non-null    float64
 6   Bodysuit                  734 non-null    float64
 7   Boots                     734 non-null    float64
 8   Bra                       734 non-null    float64
 9   Bracelet                  734 non-null    float64
 10  Cap/peaked                734 non-null    float64
 11  Cardigan                  734 non-null    float64
 12  Coat                      734 non-null    float64
 13  Costumes                  

,Bag,Ballerinas,Belt,Bikini top,Blazer,Blouse,Bodysuit,Boots,Bra,Bracelet,...,Underwear bottom,Underwear set,Vest top,Wallet,Watch,Wedge,day_of_week,month,day_of_year,is_weekend
count,734.000000,734.000000,734.000000,734.000000,734.000000,734.000000,734.000000,734.000000,734.000000,734.000000,...,734.000000,734.000000,734.00000,734.000000,734.000000,734.000000,734.000000,734.000000,734.000000,734.000000
mean,298.222071,67.136240,275.794278,1534.335150,610.527248,2050.228883,188.482289,249.205722,1819.118529,13.989101,...,1455.298365,17.072207,1926.56812,6.479564,8.897820,52.670300,3.001362,6.529973,183.444142,0.286104
std,136.746361,49.577813,125.716109,1500.232059,313.652541,983.366326,97.746486,219.658963,755.994812,9.780393,...,549.041436,21.319603,1335.41341,5.107733,7.768947,62.182895,2.002386,3.448479,105.321500,0.452247
min,72.000000,9.000000,29.000000,68.000000,137.000000,446.000000,47.000000,23.000000,599.000000,0.000000,...,437.000000,0.000000,334.00000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000
25%,216.250000,36.000000,205.250000,345.000000,447.000000,1458.750000,130.000000,75.250000,1346.000000,8.000000,...,1143.750000,3.000000,1011.75000,3.000000,4.000000,8.000000,1.000000,4.000000,92.250000,0.000000
50%,285.000000,58.000000,263.000000,1066.500000,569.000000,1784.500000,168.000000,235.000000,1687.500000,12.000000,...,1363.000000,11.000000,1433.00000,5.000000,7.000000,23.000000,3.000000,7.000000,184.000000,0.000000
75%,347.750000,84.000000,329.750000,2281.750000,712.250000,2436.000000,216.000000,355.750000,2073.750000,19.000000,...,1626.750000,24.000000,2450.25000,8.000000,11.000000,86.000000,5.000000,9.750000,273.750000,1.000000
max,1372.000000,451.000000,1718.000000,12165.000000,5123.000000,9450.000000,1102.000000,2206.000000,8721.000000,144.000000,...,6872.000000,315.000000,10486.00000,44.000000,65.000000,472.000000,6.000000,12.000000,365.000000,1.000000


<div style="background-color:#f0f2f6; padding:15px; border-radius:10px; border-left: 5px solid #4CAF50;">
    <h3 style="color:#2c3e50; margin:0;">7. Time-Series Split</h3>
    <p style="color:#555; margin-top:5px;">
        Splitting the data chronologically into three sets to prevent data leakage (future values should not be used to predict past values):
    </p>
    <ul style="color:#555; margin-top:5px;">
        <li><b>Training Set (70%):</b> Used to teach the model.</li>
        <li><b>Validation Set (15%):</b> Used to tune hyperparameters and check for overfitting during training.</li>
        <li><b>Testing Set (15%):</b> Used for the final performance evaluation on unseen data.</li>
    </ul>
</div>

In [12]:
train_ratio = 0.70
val_ratio = 0.15
test_ratio = 0.15

n = len(df)
train_end = int(n * train_ratio)
val_end = int(n * (train_ratio + val_ratio))

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

print("📊 Data Split Summary:")
print("="*60)
print(f"🎯 Training Set:   {len(train_df):>4} days ({len(train_df)/n*100:.1f}%) | From {train_df.index.min().date()} to {train_df.index.max().date()}")
print(f"🎯 Validation Set: {len(val_df):>4} days ({len(val_df)/n*100:.1f}%) | From {val_df.index.min().date()} to {val_df.index.max().date()}")
print(f"🎯 Testing Set:    {len(test_df):>4} days ({len(test_df)/n*100:.1f}%) | From {test_df.index.min().date()} to {test_df.index.max().date()}")
print("="*60)

📊 Data Split Summary:
🎯 Training Set:    513 days (69.9%) | From 2018-09-20 to 2020-02-14
🎯 Validation Set:  110 days (15.0%) | From 2020-02-15 to 2020-06-03
🎯 Testing Set:     111 days (15.1%) | From 2020-06-04 to 2020-09-22


<div style="background-color:#f0f2f6; padding:15px; border-radius:10px; border-left: 5px solid #4CAF50;">
    <h3 style="color:#2c3e50; margin:0;">8. Feature Scaling (Standardization)</h3>
    <p style="color:#555; margin-top:5px;">
        Normalizing the data is crucial for neural networks to converge efficiently.
        We use <code>StandardScaler</code> to transform the sales data for each product to have a mean of 0 and a standard deviation of 1.
        <br><b>Important:</b> The scaler is fitted <i>only</i> on the training set to avoid data leakage, and then applied to transform the validation and test sets.
    </p>
</div>

In [13]:
product_columns = [col for col in df.columns if col not in ['day_of_week', 'month', 'day_of_year', 'is_weekend']]

scalers = {}

train_scaled = train_df.copy()
val_scaled = val_df.copy()
test_scaled = test_df.copy()

print("⚙️ Normalizing data...")
print("="*60)

for col in product_columns:
    scaler = StandardScaler()

    scaler.fit(train_df[[col]])

    train_scaled[col] = scaler.transform(train_df[[col]])
    val_scaled[col] = scaler.transform(val_df[[col]])
    test_scaled[col] = scaler.transform(test_df[[col]])

    scalers[col] = scaler

print(f"✅ Successfully normalized {len(product_columns)} products!")
print("="*60)

⚙️ Normalizing data...
✅ Successfully normalized 77 products!


<div style="background-color:#f0f2f6; padding:15px; border-radius:10px; border-left: 5px solid #4CAF50;">
    <h3 style="color:#2c3e50; margin:0;">9. Save Scalers</h3>
    <p style="color:#555; margin-top:5px;">
        Saving the dictionary of scaler objects to a pickle file using <b>Joblib</b>.
        <br>This file is essential for the inference stage, as it allows us to reverse the normalization (Inverse Scaling) and interpret the model's predictions in real sales units.
    </p>
</div>

In [14]:
scalers_path = '/content/drive/MyDrive/Demand_Forecast/Scalers/scalers.pkl'
joblib.dump(scalers, scalers_path)
print(f"💾 Scalers saved at: {scalers_path}")

💾 Scalers saved at: /content/drive/MyDrive/Demand_Forecast/Scalers/scalers.pkl


<div style="background-color:#f0f2f6; padding:15px; border-radius:10px; border-left: 5px solid #4CAF50;">
    <h3 style="color:#2c3e50; margin:0;">10. Data Reshaping (Wide to Long)</h3>
    <p style="color:#555; margin-top:5px;">
        Transforming the dataset from a wide format (one column per product) to a long format (one row per timestamp-product pair).
        
        This is the required input structure for the NeuralForecast library. We use pandas <code>melt</code> for efficiency and rename columns to the standard <code>ds</code> (datestamp) and <code>y</code> (target value) format, while preserving the engineered features like day of week and month.
    </p>
</div>

In [15]:
def prepare_data_for_neuralforecast(df, product_columns):
    feature_cols = ['month', 'day_of_week', 'day_of_year', 'is_weekend']

    df_reset = df.reset_index()

    long_df = df_reset.melt(
        id_vars=['Date'] + feature_cols,
        value_vars=product_columns,
        var_name='unique_id',
        value_name='y'
    )

    long_df = long_df.rename(columns={'Date': 'ds'})

    return long_df[['ds', 'unique_id', 'y'] + feature_cols]

print("🔄 Transforming data format...")
train_nf = prepare_data_for_neuralforecast(train_scaled, product_columns)
val_nf = prepare_data_for_neuralforecast(val_scaled, product_columns)
test_nf = prepare_data_for_neuralforecast(test_scaled, product_columns)
print("✅ Transformation complete!")

🔄 Transforming data format...
✅ Transformation complete!


<div style="background-color:#f0f2f6; padding:15px; border-radius:10px; border-left: 5px solid #4CAF50;">
    <h3 style="color:#2c3e50; margin:0;">11. Model Architecture & Configuration</h3>
    <p style="color:#555; margin-top:5px;">
        Defining the <b>PatchTST</b> model structure. Key configurations include:
    </p>
    <ul style="color:#555; margin-top:5px;">
        <li><b>Horizon (7):</b> Predicting 7 days into the future.</li>
        <li><b>Input Size (180):</b> Looking back at the last 180 days of history.</li>
        <li><b>Patching:</b> Dividing the time series into small patches (length 16) to capture local semantics, similar to how Vision Transformers work.</li>
        <li><b>RevIN:</b> Reversible Instance Normalization is enabled to handle distribution shifts and non-stationary data.</li>
        <li><b>Regularization:</b> Dropout layers and weight decay are applied to prevent overfitting during the 5,000 training steps.</li>

In [16]:
horizon = 7
input_size = 365
models = [
    PatchTST(
        h=horizon,
        input_size=input_size,
        patch_len=8,
        stride=4,
        revin=True,
        hidden_size=256,
        n_heads=8,
        encoder_layers=4,
        dropout=0.1,
        fc_dropout=0.1,
        head_dropout=0.0,
        batch_size=32,
        max_steps=10000,
        learning_rate=0.0003,
        scaler_type='standard',
        loss=MAE(),
        random_seed=42,
        alias='PatchTST',
        optimizer_kwargs={'weight_decay': 1e-4},
        early_stop_patience_steps=50,
    )
]

nf = NeuralForecast(models=models, freq='D')
print("✅ Model built successfully (180-day lookback window configured)!")

INFO:lightning_fabric.utilities.seed:Seed set to 42


✅ Model built successfully (180-day lookback window configured)!


<div style="background-color:#f0f2f6; padding:15px; border-radius:10px; border-left: 5px solid #4CAF50;">
    <h3 style="color:#2c3e50; margin:0;">12. Model Training (Fitting)</h3>
    <p style="color:#555; margin-top:5px;">
        Executing the main training loop. We record the start time to track performance.
        The <code>fit</code> method takes the training data and a <code>val_size</code> argument (calculated as the number of validation timestamps per product) to monitor validation loss and trigger early stopping if the model stops improving.
    </p>
</div>

In [17]:
start_time = time.time()
print("🚀 Training started...")
print(f"⏰ Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

try:
    nf.fit(
        df=train_nf,
        val_size=len(val_nf) // len(product_columns)
    )

    elapsed_time = time.time() - start_time

    print("\n" + "="*80)
    print("✅ Training completed successfully!")
    print(f"⏱️ Elapsed time: {elapsed_time/60:.2f} minutes ({elapsed_time:.1f} seconds)")
    print(f"⏰ End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*80)

except Exception as e:
    print(f"\n❌ Error during training: {str(e)}")
    raise

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


🚀 Training started...
⏰ Time: 2026-04-14 16:57:17


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss         │ MAE               │      0 │ train │     0 │
│ 1 │ padder_train │ ConstantPad1d     │      0 │ train │     0 │
│ 2 │ scaler       │ TemporalNorm      │      0 │ train │     0 │
│ 3 │ model        │ PatchTST_backbone │  1.8 M │ train │     0 │
└───┴──────────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 1.8 M                                                                                            
Non-trainable params: 4                                                                                            
Total params: 1.8 M                                                                                                
Total estimated model params size (MB): 7                                                                          
Modules in train mode: 115                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=10000` reached.



✅ Training completed successfully!
⏱️ Elapsed time: 138.23 minutes (8293.8 seconds)
⏰ End time: 2026-04-14 19:15:31


<div style="background-color:#f0f2f6; padding:15px; border-radius:10px; border-left: 5px solid #4CAF50;">
    <h3 style="color:#2c3e50; margin:0;">13. Save Model Artifacts</h3>
    <p style="color:#555; margin-top:5px;">
        Saving the trained <b>PatchTST</b> model to Google Drive.
        We ensure the directory exists before saving. Setting <code>overwrite=True</code> ensures we always update the saved model with the latest training results.
    </p>
</div>

In [18]:
import os
save_dir = '/content/drive/MyDrive/Demand_Forecast/Models/'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

nf.save(path=save_dir, save_dataset=False, overwrite=True)
print(f"💾 Model saved successfully at: {save_dir}")

💾 Model saved successfully at: /content/drive/MyDrive/Demand_Forecast/Models/
